# 01 - Exploratory analysis
Goal: understand the imbalance, what separates fraud from normal, missing data, drift over time and leakage risks **before** modelling.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # run the notebook from /notebooks
warnings.filterwarnings("ignore")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import average_precision_score

from src import config
from src.features import load_data, add_features, time_split, NUM, CAT

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## 1. Shape and types

In [ ]:
df = load_data()
print(df.shape, '|', df.timestamp.min(), '->', df.timestamp.max())
df.head()

In [ ]:
df.dtypes

## 2. Class balance
Accuracy is useless here, so we will use **PR-AUC** as the main metric.

In [ ]:
vc = df[config.TARGET].value_counts()
print(vc, f"\nfraud rate: {df[config.TARGET].mean():.2%}  (1 in {int(1/df[config.TARGET].mean())})")
print(f"total fraud amount: {df.loc[df.is_fraud==1,'amount'].sum():,.0f}  |  median fraud amount {df.loc[df.is_fraud==1,'amount'].median():.1f} vs normal {df.loc[df.is_fraud==0,'amount'].median():.1f}")

## 3. Missing values
Check whether *being missing* is itself informative (our preprocessing adds missing-indicators).

In [ ]:
miss = df.isna().mean().sort_values(ascending=False)
print(miss[miss > 0].round(3))
for col in miss[miss > 0].index:
    print(f"{col:24s} fraud rate when missing: {df.loc[df[col].isna(), config.TARGET].mean():.2%} | present: {df.loc[df[col].notna(), config.TARGET].mean():.2%}")

## 4. Numeric features by class

In [ ]:
df = add_features(df)
df.groupby(config.TARGET)[NUM].median().T.round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, col in zip(axes.ravel(), ["log_amount", "amount_to_avg_ratio", "account_age_days", "txn_count_24h", "distance_from_home_km", "customer_age"]):
    x = df[col].clip(upper=df[col].quantile(0.99))
    for label, g in df.groupby(config.TARGET):
        sns.kdeplot(x[g.index], ax=ax, label=f"fraud={label}", common_norm=False, fill=True, alpha=.3)
    ax.set_title(col); ax.legend()
plt.tight_layout()

## 5. Categorical features: fraud rate per level
Bar = fraud rate, label = number of transactions (small groups are noisy).

In [ ]:
overall = df[config.TARGET].mean()
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col in zip(axes, CAT):
    t = df.groupby(df[col].fillna("missing"))[config.TARGET].agg(["mean", "size"]).sort_values("mean", ascending=False)
    ax.bar(t.index.astype(str), t["mean"] * 100)
    ax.axhline(overall * 100, c="red", ls="--")
    for i, n in enumerate(t["size"]): ax.text(i, 0, f"n={n}", ha="center", va="bottom", fontsize=7, rotation=90)
    ax.set_title(col); ax.set_ylabel("fraud %"); plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
plt.tight_layout()

## 6. Time patterns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
df.groupby("hour")[config.TARGET].mean().mul(100).plot(kind="bar", ax=axes[0], title="fraud % by hour")
df["acct_bucket"] = pd.cut(df.account_age_days, [0, 30, 90, 365, 1000, 1e9])
df.groupby("acct_bucket", observed=True)[config.TARGET].mean().mul(100).plot(kind="bar", ax=axes[1], title="fraud % by account age (days)")
df["ratio_bucket"] = pd.cut(df.amount_to_avg_ratio, [0, .5, 1, 2, 3, 5, 1e9])
df.groupby("ratio_bucket", observed=True)[config.TARGET].mean().mul(100).plot(kind="bar", ax=axes[2], title="fraud % by amount / 30d average")
plt.tight_layout()

In [ ]:
weekly = df.set_index("timestamp")[config.TARGET].resample("W").agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(weekly.index, weekly["mean"] * 100, marker="o"); ax.set_ylabel("fraud % (weekly)")
ax2 = ax.twinx(); ax2.bar(weekly.index, weekly["size"], alpha=.15, width=5); ax2.set_ylabel("transactions")
ax.set_title("Is the fraud rate stable over time? (drift check)")

## 7. Correlations and single-feature signal
PR-AUC of each numeric feature used alone as a score (sign-flipped if negative) vs the base rate.

In [ ]:
corr = df[NUM + [config.TARGET]].corr()[config.TARGET].drop(config.TARGET).sort_values()
corr.plot(kind="barh", figsize=(6, 5), title="correlation with is_fraud")

In [ ]:
rows = []
for col in NUM:
    x = df[col].fillna(df[col].median())
    a = average_precision_score(df[config.TARGET], x); b = average_precision_score(df[config.TARGET], -x)
    rows.append((col, max(a, b), "+" if a >= b else "-"))
pd.DataFrame(rows, columns=["feature", "pr_auc_alone", "direction"]).sort_values("pr_auc_alone", ascending=False).round(3).assign(base_rate=round(df[config.TARGET].mean(), 3))

## 8. Leakage / split sanity
We split by **time** (train on the past, test on the future). Check how many test customers were already seen in training and how the fraud rate differs.

In [ ]:
train, test = time_split(df)
print(f"train {len(train):,} ({train.is_fraud.mean():.2%} fraud)  |  test {len(test):,} ({test.is_fraud.mean():.2%} fraud)")
print(f"test transactions from customers seen in train: {test.customer_id.isin(train.customer_id).mean():.1%}")
print(f"customers with >1 fraud: {(df[df.is_fraud==1].customer_id.value_counts() > 1).sum()} of {df[df.is_fraud==1].customer_id.nunique()}")

## Takeaways (fill in for your data)
- Imbalance level and why PR-AUC / cost-based evaluation is needed
- Strongest signals found above
- Whether missingness or drift matters
- Features worth engineering next